# SOFA 对齐评估：人工标注 vs 机器标注（demucs vs mel-band-roformer）

复用 `xunmeng_SOFAevaluation.ipynb` 的逐字比较逻辑，对 **demucs** 与 **roformer** 两套机器标注分别评估，
并在最后做并排对比，判断哪种人声分离更利于强制对齐。

> 本 notebook 放在 `kunqu/` 仓库根目录（与 `SOFA/` 同级）。下方会自动向上查找
> 仓库根目录，因此无论从哪个工作目录启动 Jupyter 都能正确解析数据路径。

In [ ]:
import os, json
from pathlib import Path
import numpy as np
import pandas as pd
from collections import defaultdict
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display

plt.rcParams['font.sans-serif'] = ['PingFang SC', 'Heiti SC', 'Songti SC',
                                   'Arial Unicode MS', 'DejaVu Sans']
plt.rcParams['axes.unicode_minus'] = False
sns.set_theme(style='whitegrid', context='talk')

# ============ 自动定位仓库根目录 (kunqu/) ============
# 从当前工作目录及其父目录向上查找包含 SOFA/data/xunmeng 的目录，
# 因此 notebook 放在 kunqu/ 下、或从外层目录运行都能正确解析路径。
_REL = Path('SOFA/data/xunmeng/xunmeng_annotation.json')

def _find_root():
    for base in [Path.cwd(), *Path.cwd().parents]:
        if (base / _REL).exists():
            return base
        if (base / 'kunqu' / _REL).exists():
            return base / 'kunqu'
    raise FileNotFoundError(
        '找不到 SOFA/data/xunmeng；请在 kunqu/ 仓库内运行，或手动设置 ROOT。')

ROOT = _find_root()
print('ROOT =', ROOT)

# ================= 配置 =================
GROUND_TRUTH = str(ROOT / 'SOFA/data/xunmeng/xunmeng_annotation.json')   # 人工标注 (ground truth)
MODELS = {
    'demucs':   str(ROOT / 'SOFA/data/xunmeng/xunmeng_demucs_annotation.json'),
    'roformer': str(ROOT / 'SOFA/data/xunmeng/xunmeng_roformer_annotation.json'),
}
OUTPUT_DIR = str(ROOT / 'sofa_eval_outputs')
os.makedirs(OUTPUT_DIR, exist_ok=True)
print('输出目录:', OUTPUT_DIR)

## 1. 加载与逐字对齐（与原 notebook 完全一致的逻辑）

In [ ]:
def load_grouped(path, key='characterAnnotations'):
    """加载 JSON，按 lineId 分组并按 startTime 排序。"""
    with open(path, 'r', encoding='utf-8') as f:
        data = json.load(f)
    grouped = defaultdict(list)
    for item in data['project'][key]:
        grouped[item['lineId']].append(item)
    for lid in grouped:
        grouped[lid].sort(key=lambda x: x['startTime'])
    return data, grouped


def line_sort_key(line_id):
    # 'line-37' -> 37
    return int(line_id.split('-')[1])


def build_aligned_df(manual_grouped, machine_grouped):
    """逐句、逐字 (按下标 zip) 对齐人工与机器标注，计算 start/end 误差。"""
    rows = []
    all_ids = sorted(set(list(manual_grouped) + list(machine_grouped)), key=line_sort_key)
    for lid in all_ids:
        M = manual_grouped.get(lid, [])
        G = machine_grouped.get(lid, [])
        for i in range(max(len(M), len(G))):
            row = {'lineId': lid, 'char_index': i + 1}
            if i < len(M):
                row['manual_char']  = M[i].get('char', '')
                row['manual_start'] = M[i].get('startTime')
                row['manual_end']   = M[i].get('endTime')
            else:
                row['manual_char'] = '-缺字-'
            if i < len(G):
                row['machine_char']   = G[i].get('char', '')
                row['machine_pinyin'] = G[i].get('pinyin', '')
                row['machine_start']  = G[i].get('startTime')
                row['machine_end']    = G[i].get('endTime')
            else:
                row['machine_char'] = '-缺字-'
                row['machine_pinyin'] = '-'
            if i < len(M) and i < len(G):
                row['start_diff'] = abs(row['machine_start'] - row['manual_start'])
                row['end_diff']   = abs(row['machine_end']   - row['manual_end'])
            rows.append(row)
    cols = ['lineId', 'char_index', 'manual_char', 'machine_char', 'machine_pinyin',
            'manual_start', 'machine_start', 'start_diff',
            'manual_end', 'machine_end', 'end_diff']
    df = pd.DataFrame(rows)
    return df[[c for c in cols if c in df.columns]]

In [ ]:
# 人工标注只需加载一次
gt_data, manual_grouped = load_grouped(GROUND_TRUTH)
line_text_map = {l['id']: l['text'] for l in gt_data['project']['subtitleLines']}
print('Ground truth: %d 句, %d 字' % (
    len(gt_data['project']['subtitleLines']),
    len(gt_data['project']['characterAnnotations'])))

dfs = {}
for name, path in MODELS.items():
    _, machine_grouped = load_grouped(path)
    df = build_aligned_df(manual_grouped, machine_grouped)
    out_csv = os.path.join(OUTPUT_DIR, f'{name}_aligned_comparison.csv')
    df.to_csv(out_csv, index=False, encoding='utf-8-sig')
    dfs[name] = df
    n_pair = int(df['end_diff'].notna().sum())
    print(f'[{name}] 共 {len(df)} 行, 有效配对 N = {n_pair}  ->  {out_csv}')

display(dfs['roformer'].head(10))

## 2. 各模型统计表（并导出 LaTeX tabular，与原 notebook 一致）

In [ ]:
def stats_table(df):
    v = df.dropna(subset=['end_diff']).copy()
    s = v['start_diff']; e = v['end_diff']
    c = pd.concat([s, e], ignore_index=True)
    both01 = ((s < 0.1) & (e < 0.1)); both05 = ((s < 0.5) & (e < 0.5))
    pct = lambda x: f"{x * 100:.2f}%"
    tbl = pd.DataFrame({
        '指标': ['平均误差 (秒)', '中位数 (秒)', '标准差 (秒)', '< 0.1秒 占比', '< 0.5秒 占比'],
        '起始时间': [f"{s.mean():.4f}", f"{s.median():.4f}", f"{s.std():.4f}",
                  pct((s < 0.1).mean()), pct((s < 0.5).mean())],
        '结束时间': [f"{e.mean():.4f}", f"{e.median():.4f}", f"{e.std():.4f}",
                  pct((e < 0.1).mean()), pct((e < 0.5).mean())],
        '合并 (起始+结束)': [f"{c.mean():.4f}", f"{c.median():.4f}", f"{c.std():.4f}",
                       pct((c < 0.1).mean()), pct((c < 0.5).mean())],
        '同时满足 (起始 且 结束)': ['—', '—', '—', pct(both01.mean()), pct(both05.mean())],
    })
    return tbl, len(e)


stats = {}
for name, df in dfs.items():
    tbl, n = stats_table(df)
    stats[name] = (tbl, n)
    print(f'================ {name}  (N = {n}) ================')
    display(tbl)
    # 与原 notebook 一致：导出裸 tabular 片段（如需独立 PDF，可自行用 ctex+booktabs 包裹后 pdflatex）
    tex_path = os.path.join(OUTPUT_DIR, f'{name}_alignment_stats.tex')
    with open(tex_path, 'w', encoding='utf-8') as f:
        f.write(tbl.to_latex(index=False, column_format='lcccc'))
    print('  -> 已保存', tex_path, '\n')

## 3. 各模型可视化（箱线图 + 累积分布）

In [ ]:
for name, df in dfs.items():
    v = df.dropna(subset=['end_diff'])
    df_box = pd.concat([
        pd.DataFrame({'type': 'startTime', 'error': v['start_diff']}),
        pd.DataFrame({'type': 'endTime',   'error': v['end_diff']}),
    ], ignore_index=True)

    fig, axes = plt.subplots(1, 2, figsize=(16, 6))
    sns.boxplot(data=df_box, x='type', y='error', hue='type', palette='Set2',
                legend=False, showfliers=False, width=0.35, ax=axes[0])
    axes[0].set_title(f'{name}: Start/End Time Error (Boxplot)')
    axes[0].set_ylabel('Absolute Error (Seconds)'); axes[0].set_xlabel('')
    axes[0].set_xlim(-0.8, 1.8)

    sns.ecdfplot(data=df_box, x='error', hue='type', linewidth=3, ax=axes[1])
    axes[1].set_title(f'{name}: Cumulative Error Ratio (Higher is Better)')
    axes[1].set_xlabel('Error Threshold (Seconds)')
    axes[1].set_ylabel('Proportion of Characters')
    axes[1].set_xlim(0, 0.5)
    axes[1].axvline(x=0.1, color='red', linestyle='--', alpha=0.7)
    axes[1].text(0.12, 0.1, '0.1s Threshold', color='red', fontsize=12)

    plt.tight_layout()
    fig.savefig(os.path.join(OUTPUT_DIR, f'{name}_alignment_plots.png'), bbox_inches='tight', dpi=300)
    fig.savefig(os.path.join(OUTPUT_DIR, f'{name}_alignment_plots.pdf'), bbox_inches='tight', dpi=300)
    plt.show()

## 4. 并排对比：demucs vs roformer

In [ ]:
def metric_series(df):
    v = df.dropna(subset=['end_diff'])
    s = v['start_diff']; e = v['end_diff']
    return pd.Series({
        'start 平均误差': s.mean(), 'start 中位数': s.median(), 'start 标准差': s.std(),
        'start <0.1s %': (s < 0.1).mean() * 100, 'start <0.5s %': (s < 0.5).mean() * 100,
        'end 平均误差': e.mean(), 'end 中位数': e.median(), 'end 标准差': e.std(),
        'end <0.1s %': (e < 0.1).mean() * 100, 'end <0.5s %': (e < 0.5).mean() * 100,
        'N (配对字数)': len(e),
    })

combined = pd.DataFrame({name: metric_series(df) for name, df in dfs.items()})
print('================ 并排对比 (行=指标, 列=模型) ================')
display(combined.round(4))
combined.round(6).to_csv(os.path.join(OUTPUT_DIR, 'combined_model_comparison.csv'), encoding='utf-8-sig')
print('已保存', os.path.join(OUTPUT_DIR, 'combined_model_comparison.csv'))

In [ ]:
# 叠加 ECDF（end-time 误差）+ 阈值命中率柱状图
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

for name, df in dfs.items():
    e = df.dropna(subset=['end_diff'])['end_diff']
    sns.ecdfplot(e, label=name, linewidth=3, ax=axes[0])
axes[0].set_title('End-time Error ECDF (Higher is Better)')
axes[0].set_xlabel('Error Threshold (Seconds)')
axes[0].set_ylabel('Proportion of Characters')
axes[0].set_xlim(0, 0.5)
axes[0].axvline(x=0.1, color='red', linestyle='--', alpha=0.7)
axes[0].legend(title='model')

rates = pd.DataFrame({
    name: {
        '<0.1s': (df.dropna(subset=['end_diff'])['end_diff'] < 0.1).mean() * 100,
        '<0.5s': (df.dropna(subset=['end_diff'])['end_diff'] < 0.5).mean() * 100,
    }
    for name, df in dfs.items()
}).T
rates.plot(kind='bar', ax=axes[1], rot=0)
axes[1].set_title('End-time Accuracy by Threshold')
axes[1].set_ylabel('% of Characters')
axes[1].legend(title='threshold')

plt.tight_layout()
fig.savefig(os.path.join(OUTPUT_DIR, 'combined_plots.png'), bbox_inches='tight', dpi=300)
fig.savefig(os.path.join(OUTPUT_DIR, 'combined_plots.pdf'), bbox_inches='tight', dpi=300)
plt.show()

## 5. 结论：哪个分离模型对齐更好

In [ ]:
print('按 end-time 误差比较（mean/median 越低越好，命中率越高越好）:')
checks = [('end 平均误差', 'low'), ('end 中位数', 'low'),
          ('end <0.1s %', 'high'), ('end <0.5s %', 'high')]
for metric, better in checks:
    vals = {name: metric_series(df)[metric] for name, df in dfs.items()}
    win = (min if better == 'low' else max)(vals, key=vals.get)
    detail = ', '.join(f'{k}={v:.4f}' for k, v in vals.items())
    print(f'  {metric:14s}: {detail}   -> 更优: {win}')

## 6. 误差排行（每个模型）：按拼音 / 按句子

In [ ]:
def join_sentence(chars):
    return ''.join(str(c) for c in chars if str(c) not in ['-缺字-', 'nan'])

for name, df in dfs.items():
    print(f'\n################   {name}   ################')

    dpv = df.dropna(subset=['end_diff', 'machine_pinyin']).copy()
    pr = dpv.groupby('machine_pinyin')['end_diff'].agg(
        出现次数='count', 平均误差='mean', 中位数误差='median', 最大误差='max').reset_index()
    pr = pr[pr['出现次数'] > 3].sort_values('平均误差', ascending=False)
    print('— 平均误差最大的 5 个拼音 (出现>3) —')
    display(pr.head(5))

    dsv = df.dropna(subset=['end_diff']).copy()
    sr = dsv.groupby('lineId').agg(
        句子内容=('manual_char', join_sentence),
        有效字数=('manual_char', 'count'),
        平均误差_秒=('end_diff', 'mean'),
        最大误差_秒=('end_diff', 'max')).reset_index()
    sr = sr.sort_values('平均误差_秒', ascending=False).round(4)
    print('— 对齐最差的 10 句 —')
    display(sr.head(10))